# Rationalising an SOS certificate

`gen_pncp` produces a positive form together with a *numerical* SOS certificate.
For a rigorous proof we need an *exact* (rational) certificate. This notebook
walks through the procedure — the same one `ppt2.solve_sos(…; fix_gram=true)`
performs internally (`_fix_gram` in `src/ppt2.jl`):

1. solve the SOS feasibility SDP and read off the Gram matrix Q,
2. zero the (n−1)(m−1)+1 smallest eigenvalues of Q — the directions that should
   vanish on the Segre variety,
3. solve the resulting linear system for *rational* coefficients,
4. double-check the rationalised polynomial is still **not** SOS, so the form
   remains positive-but-not-completely-positive.

In [ ]:
include("common.jl")
using DynamicPolynomials
using SumOfSquares

## Generate a positive, non-SOS form

Retry `gen_pncp` until the bare quadratic form `f` is not already a sum of
squares (`solve_sos` returns `false` for the un-relaxed problem).

In [ ]:
n = m = 4
rng = Xoshiro(0)
f, h = ppt2.gen_pncp(n, m; rng=rng)
for _ in 1:20
    f, h = ppt2.gen_pncp(n, m; rng=rng)
    sos, _ = ppt2.solve_sos(n, m, f, h)
    sos || break
end

## Solve the relaxed SOS SDP

Maximise δ subject to (δ·f + h·h)·(x⊗y)ᵀ(x⊗y) being SOS. Feasibility with δ > 0
means the form is positive after one level of the relaxation.

In [ ]:
model = SOSModel(Mosek.Optimizer)
set_silent(model)

@polyvar X[1:n] Y[1:m]
@variable(model, delta)

XY = kron(X, Y)
f_poly = f' * kron(XY, XY)
h_poly = h' * XY

p = delta * f_poly + (h_poly' * h_poly)
relax = (XY' * XY)
con = @constraint(model, p * relax in SOSCone())
@objective(model, Max, delta)
optimize!(model)

is_solved_and_feasible(model) && value(delta) > 1e-4

## Zero the small Gram eigenvalues and rationalise

The Gram matrix Q has (n−1)(m−1)+1 eigenvalues that are numerically zero. Force
them to exactly zero, rebuild the polynomial, then solve the linear system
`A · coeff_p = coeff_G` over the rationals to recover exact coefficients.

In [ ]:
gram = gram_matrix(con)
Q = Symmetric(Matrix(gram.Q))
b = gram.basis.monomials

vals, vecs = eigen(Q)
vals[1:(n-1)*(m-1)+1] .= 0
G = vecs * Diagonal(vals) * vecs'
G_poly = b' * G * b

basis_p = monomials(p)
basis_G = monomials(G_poly)
coeff_G = DynamicPolynomials.coefficients(G_poly)

A = zeros(BigInt, length(basis_G), length(basis_p))
for i in eachindex(basis_G), j in eachindex(basis_p)
    A[i, j] = DynamicPolynomials.coefficient(basis_p[j] * relax, basis_G[i])
end

coeff_p = rationalize.(A \ coeff_G, tol=1e-10)
p_hat = coeff_p' * basis_p;

## Verify the rational certificate is not SOS

If the rationalised polynomial `p_hat` were SOS the construction would have
collapsed to a completely-positive map. We want this to be **infeasible**
(`false`).

In [ ]:
test = SOSModel(Mosek.Optimizer)
set_silent(test)
@constraint(test, p_hat in SOSCone())
@objective(test, Min, 0.0)
optimize!(test)

is_solved_and_feasible(test)

## Shortcut: the library does all of this

`solve_sos(…, fix_gram=true)` returns `(true, p_hat)` with the rational
certificate, and `pncp_poly` wraps the full generate→verify→rationalise loop.

In [ ]:
ok, p_hat_lib = ppt2.solve_sos(n, m, f, h, 1, true)
ok